In [ ]:
# ============================================================
# Toy Cosmology: Fitting Omega_Lambda from Supernova Data
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate
from scipy.optimize import curve_fit


# ------------------------------------------------------------
# Constants
# ------------------------------------------------------------

c = 300000  # km/s
H0 = 70     # km/s/Mpc


# ------------------------------------------------------------
# Cosmology Functions
# ------------------------------------------------------------

def E(z, Omega_L):
    """
    Dimensionless expansion rate for flat universe.
    Omega_m = 1 - Omega_L
    """
    Omega_m = 1 - Omega_L
    return np.sqrt(Omega_m * (1 + z)**3 + Omega_L)


def luminosity_distance(z, Omega_L):
    """
    Luminosity distance in Mpc.
    """
    integral, _ = integrate.quad(lambda zp: 1.0 / E(zp, Omega_L), 0, z)
    return (1 + z) * (c / H0) * integral


def distance_modulus(z, Omega_L):
    """
    Distance modulus prediction.
    """
    D_L = luminosity_distance(z, Omega_L)
    return 5 * np.log10(D_L) + 25


# ------------------------------------------------------------
# Generate Toy Supernova Data
# ------------------------------------------------------------

np.random.seed(1)

true_Omega_L = 0.7

z_data = np.linspace(0.01, 1.0, 25)

mu_true = np.array([distance_modulus(z, true_Omega_L) for z in z_data])

# Add observational noise
mu_data = mu_true + np.random.normal(0, 0.2, len(mu_true))

def model_mu(z, Omega_L):
    return np.array([distance_modulus(zi, Omega_L) for zi in z])

popt, pcov = curve_fit(model_mu, z_data, mu_data, p0=[0.5])

Omega_L_fit = popt[0]

print("Best-fit Omega_Lambda =", Omega_L_fit)
print("Implied Omega_m =", 1 - Omega_L_fit)


# ------------------------------------------------------------
# Plot Results
# ------------------------------------------------------------

z_plot = np.linspace(0.01, 1.0, 100)
mu_fit = model_mu(z_plot, Omega_L_fit)

plt.scatter(z_data, mu_data, label="Supernova Data")
plt.plot(z_plot, mu_fit, label="Best Fit Model")

plt.xlabel("Redshift z")
plt.ylabel("Distance Modulus μ")
plt.title("Toy Supernova Fit for Omega_Lambda")
plt.legend()

plt.show()